In [1]:
from pathlib import Path
import glob
import re

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import rdMolAlign

# Silence RDKit logs (explicit valence errors, sanitization warnings, etc.)
RDLogger.DisableLog('rdApp.*')

In [9]:
# for pdb_path in glob.glob("sampling/train_ref/*.pdb"):
#     pdb_num = int(re.search(r'(\d+).pdb', pdb_path).group(1))
#     mol = Chem.MolFromPDBFile(pdb_path)
#     Chem.MolToPDBFile(mol, f"sampling/test_{pdb_num}.pdb")

In [2]:
from rdkit.Chem import rdchem, rdmolops


def mol_to_match_key(mol, stereo=False):
    """
    Canonical SMILES with all explicit Hs and formal charges removed.
    Remove Hs first so heavy-atom valences are correct after charge zeroing
    (e.g. [SH+] -> remove H -> S+ with 2 bonds -> charge=0 -> S, valid valence 2).
    stereo=False strips stereo annotations for connectivity-only matching.
    """
    mol = Chem.RemoveHs(mol)
    rw = Chem.RWMol(mol)
    for atom in rw.GetAtoms():
        atom.SetFormalCharge(0)
    try:
        Chem.SanitizeMol(rw)
    except Exception:
        try:
            Chem.SanitizeMol(rw, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES)
        except Exception:
            pass
    return Chem.MolToSmiles(rw.GetMol(), isomericSmiles=stereo, canonical=True)


def canon_smi_from_mol(mol, stereo=True):
    """Canonical SMILES for display — same neutralization as matching key."""
    return mol_to_match_key(mol, stereo=stereo)


def load_pdb(path, perceive_stereo=False):
    mol = Chem.MolFromPDBFile(path, removeHs=True, sanitize=False)
    if mol is None:
        return None
    if not perceive_stereo:
        for atom in mol.GetAtoms():
            atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)
        for bond in mol.GetBonds():
            bond.SetStereo(rdchem.BondStereo.STEREONONE)
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    if perceive_stereo:
        Chem.AssignStereochemistry(mol, cleanIt=False, force=True)
    return mol


def heavy_atom_rmsd(mol_ref, mol_gen):
    ref = Chem.RemoveHs(mol_ref)
    gen = Chem.RemoveHs(mol_gen)
    try:
        return rdMolAlign.GetBestRMS(ref, gen)
    except RuntimeError:
        return float('inf')


def bond_length_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    assert mol_ref.GetNumAtoms() == mol_gen.GetNumAtoms()
    assert mol_ref.GetNumBonds() == mol_gen.GetNumBonds()
    for b in mol_ref.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        assert mol_gen.GetBondBetweenAtoms(i, j) is not None

    errs = []
    for b in mol_ref.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        d_r = conf_r.GetAtomPosition(i).Distance(conf_r.GetAtomPosition(j))
        d_g = conf_g.GetAtomPosition(i).Distance(conf_g.GetAtomPosition(j))
        errs.append(abs(d_r - d_g))

    return float(np.mean(errs))


def angle(a, b, c):
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba == 0 or norm_bc == 0:
        return 0.0
    cos_angle = np.clip(np.dot(ba, bc) / (norm_ba * norm_bc), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def bond_angle_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for atom in mol_ref.GetAtoms():
        nbrs = [n.GetIdx() for n in atom.GetNeighbors()]
        if len(nbrs) < 2:
            continue
        for i in range(len(nbrs)):
            for j in range(i + 1, len(nbrs)):
                a, b, c = nbrs[i], atom.GetIdx(), nbrs[j]
                ar = angle(conf_r.GetAtomPosition(a), conf_r.GetAtomPosition(b), conf_r.GetAtomPosition(c))
                ag = angle(conf_g.GetAtomPosition(a), conf_g.GetAtomPosition(b), conf_g.GetAtomPosition(c))
                errs.append(abs(ar - ag))
    return float(np.mean(errs))


def torsion_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    torsions = []
    for bond in mol_ref.GetBonds():
        if bond.IsInRing() or bond.GetBondType() != Chem.BondType.SINGLE:
            continue

        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ai, aj = mol_ref.GetAtomWithIdx(i), mol_ref.GetAtomWithIdx(j)
        if ai.GetDegree() < 2 or aj.GetDegree() < 2:
            continue

        ni = sorted(a.GetIdx() for a in ai.GetNeighbors() if a.GetIdx() != j)
        nj = sorted(a.GetIdx() for a in aj.GetNeighbors() if a.GetIdx() != i)

        for a in ni:
            for d in nj:
                tr = rdMolTransforms.GetDihedralDeg(conf_r, a, i, j, d)
                tg = rdMolTransforms.GetDihedralDeg(conf_g, a, i, j, d)
                diff = abs(tr - tg) % 360
                torsions.append(min(diff, 360 - diff))

    return float(np.mean(torsions)) if torsions else 0.0


def heavy_atom_rmsf(mol_list, align_first=True):
    if len(mol_list) < 2:
        return 0.0

    mols_noH = [Chem.RemoveHs(mol) for mol in mol_list]
    n_atoms = mols_noH[0].GetNumAtoms()
    if not all(mol.GetNumAtoms() == n_atoms for mol in mols_noH):
        return None

    coords_list = [
        np.array([mol.GetConformer().GetAtomPosition(i) for i in range(n_atoms)])
        for mol in mols_noH
    ]
    coords_array = np.stack(coords_list, axis=0)

    if align_first:
        ref_coords = [coords_array[0]]
        for i in range(1, coords_array.shape[0]):
            mol_aligned = Chem.Mol(mols_noH[i])
            rdMolAlign.AlignMol(mol_aligned, mols_noH[0])
            conf_aligned = mol_aligned.GetConformer()
            ref_coords.append(np.array([conf_aligned.GetAtomPosition(j) for j in range(n_atoms)]))
        coords_array = np.stack(ref_coords, axis=0)

    mean_coords = np.mean(coords_array, axis=0)
    squared_deviations = np.sum((coords_array - mean_coords[None]) ** 2, axis=2)
    rmsf_per_atom = np.sqrt(np.mean(squared_deviations, axis=0))
    return float(np.mean(rmsf_per_atom))


def mmff_energy(mol):
    props = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, props)
    return ff.CalcEnergy()


def clash_count(mol, scale=0.75):
    conf = mol.GetConformer()
    pts = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    vdw = np.array([
        rdchem.GetPeriodicTable().GetRvdw(a.GetAtomicNum())
        for a in mol.GetAtoms()
    ])

    n = mol.GetNumAtoms()
    clashes = 0
    for i in range(n):
        if mol.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        for j in range(i + 1, n):
            if mol.GetAtomWithIdx(j).GetAtomicNum() == 1:
                continue
            path = rdmolops.GetShortestPath(mol, i, j)
            if path is not None and len(path) <= 4:
                continue
            if np.linalg.norm(pts[i] - pts[j]) < scale * (vdw[i] + vdw[j]):
                clashes += 1
    return clashes

In [3]:
import glob
from pathlib import Path
from collections import defaultdict
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolAlign, rdmolfiles
from joblib import Parallel, delayed
import multiprocessing as mp
import os

# ============================================================
# USER SETTINGS
# ============================================================

mode = "train"  # "train" or "test"
model_name = "geom_identityRot_256_conformer_train_3std_bondlength"

REF_GLOB = f"{Path(os.getcwd()).parent}/sampling/geom_conformer_{mode}/conformer_mols/*.pdb"
PRED_GLOB = f"{Path(os.getcwd()).parent}/sampling/geom_conformer_{mode}/{model_name}/samples/*.pdb"

OUT_ALIGN_DIR = Path(f"{Path(os.getcwd()).parent}/aligned_pairs/geom_conformer_{mode}/{model_name}")
OUT_ALIGN_DIR.mkdir(exist_ok=True, parents=True)

DELTA = 0.75   # RMSD threshold for coverage (Å)
N_JOBS = max(1, mp.cpu_count() // 2)
USE_PARALLEL = True


# ============================================================
# Utilities
# ============================================================

def load_and_prepare(pdb_path):
    mol = load_pdb(pdb_path, perceive_stereo=False)
    if mol is None:
        return None, None, None, None, None
    mol = Chem.AddHs(mol)

    display_smi_no_stereo = canon_smi_from_mol(mol, stereo=False)
    match_no_stereo = mol_to_match_key(mol, stereo=False)

    mol_stereo = load_pdb(pdb_path, perceive_stereo=True)
    if mol_stereo is not None:
        mol_stereo = Chem.AddHs(mol_stereo)
        display_smi_stereo = canon_smi_from_mol(mol_stereo, stereo=True)
        match_stereo = mol_to_match_key(mol_stereo, stereo=True)
    else:
        display_smi_stereo = display_smi_no_stereo
        match_stereo = match_no_stereo

    return mol, match_no_stereo, match_stereo, display_smi_no_stereo, display_smi_stereo


def pairwise_rmsd_matrix(ref_mols, gen_mols):
    D = np.zeros((len(ref_mols), len(gen_mols)), dtype=float)
    for i, r in enumerate(ref_mols):
        for j, g in enumerate(gen_mols):
            D[i, j] = heavy_atom_rmsd(r, g)
    return D


def compute_coverage_amr(D, delta):
    """Compute AMR-R, COV-R, AMR-P, COV-P from an RMSD matrix, ignoring inf entries."""
    if D.size == 0:
        return float('nan'), float('nan'), float('nan'), float('nan')
    min_ref = np.where(np.isfinite(D), D, np.inf).min(axis=1)
    min_gen = np.where(np.isfinite(D), D, np.inf).min(axis=0)
    valid_ref = np.isfinite(min_ref)
    valid_gen = np.isfinite(min_gen)
    amr_r = float(min_ref[valid_ref].mean()) if valid_ref.any() else float('nan')
    cov_r = float((min_ref[valid_ref] < delta).mean()) if valid_ref.any() else float('nan')
    amr_p = float(min_gen[valid_gen].mean()) if valid_gen.any() else float('nan')
    cov_p = float((min_gen[valid_gen] < delta).mean()) if valid_gen.any() else float('nan')
    return amr_r, cov_r, amr_p, cov_p


def compute_geometry_stats(ref_mols, gen_mols, gen_paths, D, out_align_dir):
    """Per-conformer geometry metrics; returns (best_rmsd, best_bl, best_ba, mean_tor, mean_clash, rmsf)."""
    per_rmsd, per_bl, per_ba, per_tor, per_clash = [], [], [], [], []

    for j, (gen, p) in enumerate(zip(gen_mols, gen_paths)):
        col = D[:, j]
        finite_rows = np.where(np.isfinite(col))[0]
        if len(finite_rows) == 0:
            continue
        i = int(finite_rows[np.argmin(col[finite_rows])])
        ref = ref_mols[i]

        per_rmsd.append(D[i, j])
        try:
            per_bl.append(bond_length_mae(gen, ref))
            per_ba.append(bond_angle_mae(gen, ref))
            per_tor.append(torsion_mae(gen, ref))
        except Exception:
            pass
        per_clash.append(clash_count(gen))

        if out_align_dir is not None:
            try:
                probe = Chem.Mol(gen)
                ref_a = Chem.Mol(ref)
                rdMolAlign.AlignMol(probe, ref_a)
                out_name = f"{p.stem}_BEST_rmsd{D[i, j]:.3f}.pdb"
                save_best_pair(probe, ref_a, out_align_dir / out_name)
            except Exception:
                pass

    if not per_rmsd:
        return None
    rmsf_val = heavy_atom_rmsf(gen_mols, align_first=True) if len(gen_mols) >= 2 else None
    return (
        float(np.min(per_rmsd)),
        float(np.min(per_bl))   if per_bl  else float('nan'),
        float(np.min(per_ba))   if per_ba  else float('nan'),
        float(np.mean(per_tor)) if per_tor else float('nan'),
        float(np.mean(per_clash)),
        rmsf_val,
    )


def compute_reference_rmsf_mean(ref_key_to_mols):
    vals = []
    for mols in ref_key_to_mols.values():
        if len(mols) < 2:
            continue
        try:
            v = heavy_atom_rmsf(mols, align_first=True)
            if v is not None and not np.isnan(v):
                vals.append(v)
        except Exception:
            pass
    return float(np.mean(vals)) if vals else None


def save_best_pair(m1, m2, out_path):
    m1 = Chem.RemoveHs(m1)
    m2 = Chem.RemoveHs(m2)
    with open(out_path, "w") as f:
        f.write("MODEL        1\n")
        f.write(rdmolfiles.MolToPDBBlock(m1, flavor=4))
        f.write("ENDMDL\n")
        f.write("MODEL        2\n")
        f.write(rdmolfiles.MolToPDBBlock(m2, flavor=4))
        f.write("ENDMDL\n")
        f.write("END\n")


# ============================================================
# Load Reference Molecules
# ============================================================

print("Loading reference molecules...")

ref_key_no_stereo_to_mols = defaultdict(list)
ref_key_stereo_to_mols = defaultdict(list)
ref_mol_id_to_display_smis = {}

for ref_path in glob.glob(REF_GLOB):
    mol, match_no_stereo, match_stereo, display_smi_no_stereo, display_smi_stereo = load_and_prepare(ref_path)
    assert mol is not None, f"Failed to load ref: {ref_path}"
    ref_key_no_stereo_to_mols[match_no_stereo].append(mol)
    ref_key_stereo_to_mols[match_stereo].append(mol)
    mol_id = Path(ref_path).stem.rsplit("_", 1)[0]
    if mol_id not in ref_mol_id_to_display_smis:
        ref_mol_id_to_display_smis[mol_id] = (display_smi_no_stereo, display_smi_stereo)

print(f"Loaded {sum(len(v) for v in ref_key_no_stereo_to_mols.values())} reference conformers")
print(f"{len(ref_key_no_stereo_to_mols)} unique keys in reference (no stereo)")
print(f"{len(ref_key_stereo_to_mols)} unique keys in reference (with stereo)")


# ============================================================
# Collect Predicted Molecules
# ============================================================

print("Collecting predicted PDBs...")

pred_paths = [Path(p) for p in glob.glob(PRED_GLOB)]

groups = defaultdict(list)
for p in pred_paths:
    mol_id = p.stem.rsplit("_", 1)[0]
    groups[mol_id].append(p)

print(f"{len(groups)} molecule groups found")


# ============================================================
# Metric Computation Per Molecule
# ============================================================

def compute_metrics_for_group(paths, ref_key_no_stereo_to_mols, ref_key_stereo_to_mols, out_align_dir):
    RDLogger.DisableLog('rdApp.*')

    gen_mols, gen_paths, gen_stereo_flags = [], [], []
    ref_mols_for_group = None
    mol_id = paths[0].stem.rsplit("_", 1)[0] if paths else None

    connectivity_no_match = []
    stereo_no_match = []
    n_attempted = 0
    n_matched_no_stereo = 0
    n_matched_stereo = 0

    for p in sorted(paths):
        mol, match_no_stereo, match_stereo, display_smi_no_stereo, display_smi_stereo = load_and_prepare(p)
        if mol is None:
            continue

        n_attempted += 1
        ref_mols = ref_key_no_stereo_to_mols.get(match_no_stereo)
        if ref_mols is None:
            connectivity_no_match.append((mol_id, display_smi_no_stereo))
            continue

        n_matched_no_stereo += 1
        stereo_ok = ref_key_stereo_to_mols.get(match_stereo) is not None
        if stereo_ok:
            n_matched_stereo += 1
        else:
            stereo_no_match.append((mol_id, display_smi_stereo))

        gen_mols.append(mol)
        gen_paths.append(p)
        gen_stereo_flags.append(stereo_ok)
        ref_mols_for_group = ref_mols

    empty = (None, None, connectivity_no_match, stereo_no_match, n_attempted, n_matched_no_stereo, n_matched_stereo)
    if not gen_mols or ref_mols_for_group is None:
        return empty

    D = pairwise_rmsd_matrix(ref_mols_for_group, gen_mols)

    # ---- connectivity-matched stats (all gen mols) ----
    amr_r, cov_r, amr_p, cov_p = compute_coverage_amr(D, DELTA)
    geom_conn = compute_geometry_stats(ref_mols_for_group, gen_mols, gen_paths, D, out_align_dir)
    if geom_conn is None:
        return empty
    metrics_conn = (amr_r, cov_r, amr_p, cov_p) + geom_conn

    # ---- stereo-matched stats (subset) ----
    stereo_idx = [j for j, ok in enumerate(gen_stereo_flags) if ok]
    if stereo_idx:
        D_stereo = D[:, stereo_idx]
        gen_mols_s = [gen_mols[j] for j in stereo_idx]
        gen_paths_s = [gen_paths[j] for j in stereo_idx]
        amr_r_s, cov_r_s, amr_p_s, cov_p_s = compute_coverage_amr(D_stereo, DELTA)
        geom_stereo = compute_geometry_stats(ref_mols_for_group, gen_mols_s, gen_paths_s, D_stereo, None)
        metrics_stereo = (amr_r_s, cov_r_s, amr_p_s, cov_p_s) + (geom_stereo if geom_stereo else (float('nan'),) * 6)
    else:
        metrics_stereo = None

    return (metrics_conn, metrics_stereo, connectivity_no_match, stereo_no_match,
            n_attempted, n_matched_no_stereo, n_matched_stereo)


# ============================================================
# Run Computation
# ============================================================

print("Running metric computation...")

if USE_PARALLEL:
    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(compute_metrics_for_group)(paths, ref_key_no_stereo_to_mols, ref_key_stereo_to_mols, OUT_ALIGN_DIR)
        for paths in groups.values()
    )
else:
    results = [
        compute_metrics_for_group(paths, ref_key_no_stereo_to_mols, ref_key_stereo_to_mols, OUT_ALIGN_DIR)
        for paths in groups.values()
    ]


# ============================================================
# Aggregate Results
# ============================================================

def make_metric_lists():
    return {k: [] for k in ['amr_r','cov_r','amr_p','cov_p','rmsd','bl','ba','tor','clash','rmsf']}

agg_conn   = make_metric_lists()
agg_stereo = make_metric_lists()
connectivity_no_match_all = []
stereo_no_match_all = []
total_attempted = 0
total_matched_no_stereo = 0
total_matched_stereo = 0

def append_metrics(agg, metrics):
    amr_r, cov_r, amr_p, cov_p, rmsd, bl, ba, tor, clash, rmsf_val = metrics
    for key, val in [('amr_r',amr_r),('cov_r',cov_r),('amr_p',amr_p),('cov_p',cov_p),
                     ('rmsd',rmsd),('bl',bl),('ba',ba),('tor',tor),('clash',clash)]:
        if np.isfinite(val):
            agg[key].append(val)
    if rmsf_val is not None:
        agg['rmsf'].append(rmsf_val)

for res in results:
    metrics_conn, metrics_stereo, conn_nm, stereo_nm, n_att, n_conn, n_stereo = res
    total_attempted += n_att
    total_matched_no_stereo += n_conn
    total_matched_stereo += n_stereo
    connectivity_no_match_all.extend(conn_nm)
    stereo_no_match_all.extend(stereo_nm)
    if metrics_conn is not None:
        append_metrics(agg_conn, metrics_conn)
    if metrics_stereo is not None:
        append_metrics(agg_stereo, metrics_stereo)

ref_rmsf_mean = compute_reference_rmsf_mean(ref_key_no_stereo_to_mols)


# ============================================================
# Summary
# ============================================================

def mean_or_nan(lst):
    return float(np.mean(lst)) if lst else float('nan')

print(f"\n==============================")
print(f"   FINAL RESULTS ({mode.upper()})")
print(f"==============================")

print(f"\n--- Generated Molecules ---")
print(f"Correctly generated — connectivity only (no stereo):  {total_matched_no_stereo}/{total_attempted}")
print(f"Correctly generated — connectivity + stereochemistry: {total_matched_stereo}/{total_attempted}")
print(f"No-match (wrong connectivity):                        {total_attempted - total_matched_no_stereo}/{total_attempted}")
print(f"No-match (correct connectivity, wrong stereo):        {total_matched_no_stereo - total_matched_stereo}/{total_attempted}")

for label, agg in [("Connectivity-matched (no stereo)", agg_conn),
                   ("Connectivity + stereo matched",    agg_stereo)]:
    print(f"\n--- {label} ---")
    print(f"  AMR-R:          {mean_or_nan(agg['amr_r']):.4f}")
    print(f"  COV-R:          {mean_or_nan(agg['cov_r']):.4f}")
    print(f"  AMR-P:          {mean_or_nan(agg['amr_p']):.4f}")
    print(f"  COV-P:          {mean_or_nan(agg['cov_p']):.4f}")
    print(f"  Best RMSD:      {mean_or_nan(agg['rmsd']):.4f}")
    print(f"  Best BL MAE:    {mean_or_nan(agg['bl']):.4f}")
    print(f"  Best BA MAE:    {mean_or_nan(agg['ba']):.4f}")
    print(f"  Torsion MAE:    {mean_or_nan(agg['tor']):.4f}")
    print(f"  Clashes:        {mean_or_nan(agg['clash']):.4f}")
    print(f"  RMSF (gen):     {mean_or_nan(agg['rmsf']):.4f}")

print(f"\n  RMSF (ref):     {ref_rmsf_mean}")


# ---- No-match: connectivity failures ----
if connectivity_no_match_all:
    print(f"\n--- No-match: wrong connectivity ({len(connectivity_no_match_all)} conformers) ---")
    by_name = defaultdict(list)
    for mol_id, gen_smi in connectivity_no_match_all:
        by_name[mol_id].append(gen_smi)
    for mol_id, gen_smis in sorted(by_name.items()):
        ref_smis = ref_mol_id_to_display_smis.get(mol_id)
        ref_smi = ref_smis[0] if ref_smis else None
        print(f"  [{mol_id}]  ({len(gen_smis)} conformers)")
        print(f"    ref: {ref_smi}")
        for smi in sorted(set(gen_smis)):
            print(f"    gen: {smi}")
        print()
else:
    print("\n--- No-match (connectivity): none ---")

# ---- No-match: stereo failures ----
if stereo_no_match_all:
    print(f"\n--- No-match: wrong stereo, correct connectivity ({len(stereo_no_match_all)} conformers) ---")
    by_name = defaultdict(list)
    for mol_id, gen_smi_stereo in stereo_no_match_all:
        by_name[mol_id].append(gen_smi_stereo)
    for mol_id, gen_smis in sorted(by_name.items()):
        ref_smis = ref_mol_id_to_display_smis.get(mol_id)
        ref_smi_stereo = ref_smis[1] if ref_smis else None
        print(f"  [{mol_id}]  ({len(gen_smis)} conformers)")
        print(f"    ref: {ref_smi_stereo}")
        for smi in sorted(set(gen_smis)):
            print(f"    gen: {smi}")
        print()
else:
    print("\n--- No-match (stereo): none ---")

# ---- Perfectly matched molecules (no connectivity or stereo failures) ----
no_match_mol_ids = set(mol_id for mol_id, _ in connectivity_no_match_all) |                    set(mol_id for mol_id, _ in stereo_no_match_all)
perfect = [(mol_id, ref_mol_id_to_display_smis[mol_id][0])
           for mol_id in sorted(groups.keys())
           if mol_id not in no_match_mol_ids]
if perfect:
    print(f"\n--- Perfectly matched molecules ({len(perfect)}/{len(groups)}, all conformers correct connectivity + stereo) ---")
    for mol_id, smi in perfect:
        print(f"  [{mol_id}]  {smi}")
else:
    print("\n--- Perfectly matched molecules: none ---")

Loading reference molecules...
Loaded 6236 reference conformers
30 unique keys in reference (no stereo)
337 unique keys in reference (with stereo)
30 molecule groups found
Running metric computation...

   FINAL RESULTS (TRAIN)

--- Generated Molecules ---
Correctly generated — connectivity only (no stereo):  150/150
Correctly generated — connectivity + stereochemistry: 75/150
No-match (wrong connectivity):                        0/150
No-match (correct connectivity, wrong stereo):        75/150

--- Connectivity-matched (no stereo) ---
  AMR-R:          1.8579
  COV-R:          0.0077
  AMR-P:          1.3923
  COV-P:          0.0333
  Best RMSD:      1.0532
  Best BL MAE:    0.3461
  Best BA MAE:    12.8453
  Torsion MAE:    51.3338
  Clashes:        0.0000
  RMSF (gen):     1.4480

--- Connectivity + stereo matched ---
  AMR-R:          1.8187
  COV-R:          0.0031
  AMR-P:          1.4077
  COV-P:          0.0333
  Best RMSD:      1.1058
  Best BL MAE:    0.3402
  Best BA MAE:  